# Обучение модели HistGradientBoosting для задачи бинарной классификации - предсказания купит ли пользователь товар

In [1]:
import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder

from purchase_intent.config import settings
from purchase_intent.features import add_features

In [2]:
SEED = 42
TEST_SIZE = 0.20

DATA_PATH = settings.data_dir / "online_shoppers_intention.csv"
ARTIFACT_PATH = settings.model_path

ARTIFACT_VERSION = "2.0.0"

CLASSIFICATION_THRESHOLD = 0.27

HGB_PARAMS = {
    "learning_rate": 0.05,
    "max_depth": 6,
    "random_state": SEED,
}

## 1. Загрузка данных

In [3]:
df = pd.read_csv(DATA_PATH)

TARGET = "Revenue"

df = df.drop_duplicates().reset_index(drop=True)

df[TARGET] = df[TARGET].astype(int)

X = df.drop(columns=TARGET)
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=SEED
)

print(f"Всего объектов после удаления дубликатов: {len(df)}")
print(f"Train: {X_train.shape}")
print(f"Test:  {X_test.shape}")
print(f"Доля класса 1: train={y_train.mean():.3f}, test={y_test.mean():.3f}")

Всего объектов после удаления дубликатов: 12205
Train: (9764, 17)
Test:  (2441, 17)
Доля класса 1: train=0.156, test=0.156


## 2. Feature engineering

Функция `add_features` лежит в папке (src/purchase_intent/features.py), а не в ноутбуке: joblib сохраняет функцию как ссылку "модуль + имя", и сервис должен найти ее по этой ссылке при загрузке бандла. Функция также приводит типы: коды категорий в строки, Weekend в 0/1, поэтому данные из CSV и из запроса к сервису проходят одинаковую обработку.

In [4]:
FE = FunctionTransformer(add_features)

new_cols = [c for c in add_features(X_train.head()).columns if c not in X_train.columns]
add_features(X_train)[new_cols].describe().T.round(3)

,count,mean,std,min,25%,50%,75%,max
has_PageValues,9764.0,0.224,0.417,0.0,0.000,0.000,0.000,1.0
has_Informational,9764.0,0.216,0.411,0.0,0.000,0.000,0.000,1.0
has_BounceRates,9764.0,0.547,0.498,0.0,0.000,1.000,1.000,1.0
has_SpecialDay,9764.0,0.103,0.304,0.0,0.000,0.000,0.000,1.0
Administrative_avg_time,9764.0,20.563,47.122,0.0,0.000,5.793,24.504,1566.6
Informational_avg_time,9764.0,14.884,65.985,0.0,0.000,0.000,0.000,1778.0
ProductRelated_avg_time,9764.0,38.378,46.986,0.0,17.967,29.290,45.666,1411.0


## 3. Предобработка

In [5]:
LOG_COLS = [
    "Administrative",
    "ProductRelated",
    "Administrative_avg_time",
    "Informational_avg_time",
    "ProductRelated_avg_time",
]

NUM_COLS = ["ExitRates"]

CAT_COLS = [
    "Month",
    "VisitorType",
    "OperatingSystems",
    "Browser",
    "Region",
    "TrafficType",
]

BIN_COLS = [
    "Weekend",
    "has_PageValues",
    "has_Informational",
    "has_SpecialDay",
    "has_BounceRates",
]


def make_preprocessor():
    log_pipeline = Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            ("log1p", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
        ]
    )

    num_pipeline = Pipeline([("imputer", SimpleImputer(strategy="median"))])

    categorical_pipeline = Pipeline(
        [
            ("imputer", SimpleImputer(strategy="most_frequent")),
            (
                "ohe",
                OneHotEncoder(
                    handle_unknown="infrequent_if_exist",
                    min_frequency=0.01,
                    sparse_output=False,
                ),
            ),
        ]
    )

    return ColumnTransformer(
        transformers=[
            ("log_num", log_pipeline, LOG_COLS),
            ("num", num_pipeline, NUM_COLS),
            ("cat", categorical_pipeline, CAT_COLS),
            ("bin", "passthrough", BIN_COLS),
        ],
        remainder="drop",
    )

## 4. Pipeline и HistGradientBoosting

In [6]:
pipeline = Pipeline(
    [
        ("fe", FE),
        ("preprocessing", make_preprocessor()),
        ("clf", HistGradientBoostingClassifier(**HGB_PARAMS)),
    ]
)

pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('fe', ...), ('preprocessing', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"func func: callable, default=NoneThe callable to use for the transformation. This will be passedthe same arguments as transform, with args and kwargs forwarded.If func is None, then func will be the identity function.",<function add...001D1D2D60040>
,"inverse_func inverse_func: callable, default=NoneThe callable to use for the inverse transformation. This will bepassed the same arguments as inverse transform, with args andkwargs forwarded. If inverse_func is None, then inverse_funcwill be the identity function.",None
,"validate validate: bool, default=FalseIndicate that the input X array should be checked before calling``func``. The possibilities are:- If False, there is no input validation.- If True, then X will be converted to a 2-dimensional NumPy array or sparse matrix. If the conversion is not possible an exception is raised... versionchanged:: 0.22 The default of ``validate`` changed from True to False.",False
,"accept_sparse accept_sparse: bool, default=FalseIndicate that func accepts a sparse matrix as input. If validate isFalse, this has no effect. Otherwise, if accept_sparse is false,sparse matrix inputs will cause an exception to be raised.",False
,"check_inverse check_inverse: bool, default=TrueWhether to check that or ``func`` followed by ``inverse_func`` leads tothe original inputs. It can be used for a sanity check, raising awarning when the condition is not fulfilled... versionadded:: 0.20",True
,"feature_names_out feature_names_out: callable, 'one-to-one' or None, default=NoneDetermines the list of feature names that will be returned by the`get_feature_names_out` method. If it is 'one-to-one', then the outputfeature names will be equal to the input feature names. If it is acallable, then it must take two positional arguments: this`FunctionTransformer` (`self`) and an array-like of input feature names(`input_features`). It must return an array-like of output featurenames. The `get_feature_names_out` method is only defined if`feature_names_out` is not None.See ``get_feature_names_out`` for more details... versionadded:: 1.1",None
,"kw_args kw_args: dict, default=NoneDictiona

## 5. Обучение

In [7]:
pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('fe', ...), ('preprocessing', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](17,)","['Administrative','Administrative_Duration','Informational',..., 'TrafficType','VisitorType','Weekend']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,17
,"func func: callable, default=NoneThe callable to use for the transformation. This will be passedthe same arguments as transform, with args and kwargs forwarded.If func is None, then func will be the identity function.",<function add...001D1D2D60040>
,"inverse_func inverse_func: callable, default=NoneThe callable to use for the inverse transformation. This will bepassed the same arguments as inverse transform, with args andkwargs forwarded. If inverse_func is None, then inverse_funcwill be the identity function.",None
,"validate validate: bool, default=FalseIndicate that the input X array should be checked before calling``func``. The possibilities are:- If False, there is no input validation.- If True, then X will be converted to a 2-dimensional NumPy array or sparse matrix. If the conversion is not possible an exception is raised... versionchanged:: 0.22 The default of ``validate`` changed from True to False.",False


## 6. Финальная проверка на тестовой выборке

In [8]:
test_proba = pipeline.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= CLASSIFICATION_THRESHOLD).astype(int)

test_metrics = {
    "average_precision": average_precision_score(y_test, test_proba),
    "roc_auc": roc_auc_score(y_test, test_proba),
    "f1": f1_score(y_test, test_pred),
    "precision": precision_score(y_test, test_pred),
    "recall": recall_score(y_test, test_pred),
    "accuracy": accuracy_score(y_test, test_pred),
}

pd.Series(test_metrics).round(4)

average_precision    0.7555
roc_auc              0.9351
f1                   0.6798
precision            0.6087
recall               0.7696
accuracy             0.8865
dtype: float64

In [9]:
print(
    classification_report(
        y_test,
        test_pred,
        target_names=["нет покупки", "покупка"],
        digits=4,
    )
)

              precision    recall  f1-score   support

 нет покупки     0.9551    0.9082    0.9310      2059
     покупка     0.6087    0.7696    0.6798       382

    accuracy                         0.8865      2441
   macro avg     0.7819    0.8389    0.8054      2441
weighted avg     0.9009    0.8865    0.8917      2441



In [10]:
cm = confusion_matrix(y_test, test_pred)

print("Матрица ошибок:")
cm

Матрица ошибок:


array([[1870,  189],
       [  88,  294]])

## 7. Metadata

In [11]:
FEATURES = [
    "Administrative",
    "Administrative_Duration",
    "Informational",
    "Informational_Duration",
    "ProductRelated",
    "ProductRelated_Duration",
    "BounceRates",
    "ExitRates",
    "PageValues",
    "SpecialDay",
    "Month",
    "OperatingSystems",
    "Browser",
    "Region",
    "TrafficType",
    "VisitorType",
    "Weekend",
]

assert FEATURES == list(X_train.columns), (
    "Порядок входных признаков изменился. Проверьте схему исходных данных."
)

assert pipeline["fe"].func.__module__ == "purchase_intent.features", (
    "функция add_features должна импортироваться из пакета, иначе сервис не загрузит бандл."
)

metadata = {
    "model_version": ARTIFACT_VERSION,
    "features": FEATURES,
    "threshold": CLASSIFICATION_THRESHOLD,
}

metadata

{'model_version': '2.0.0',
 'features': ['Administrative',
  'Administrative_Duration',
  'Informational',
  'Informational_Duration',
  'ProductRelated',
  'ProductRelated_Duration',
  'BounceRates',
  'ExitRates',
  'PageValues',
  'SpecialDay',
  'Month',
  'OperatingSystems',
  'Browser',
  'Region',
  'TrafficType',
  'VisitorType',
  'Weekend'],
 'threshold': 0.27}

## 8. Сохранение joblib-бандла

In [12]:
bundle = {
    "pipeline": pipeline,
    "metadata": metadata,
}

ARTIFACT_PATH.parent.mkdir(exist_ok=True)
joblib.dump(bundle, ARTIFACT_PATH)

print(f"Артефакт сохранен: {ARTIFACT_PATH}")

Артефакт сохранен: D:\PycharmProjects\purchase-intent-service\artifact\model.joblib
